# sage-proteomics基礎：DIA-MSの原理とセットアップ【論文再現シリーズ #4a】

## はじめに

この記事では、DIA-MSの生データからタンパク質を同定・定量する方法を解説します。論文（Toyota et al. 2025）では **DIA-NN v1.8.1** を使用していますが、DIA-NN は**商用利用に有料ライセンスが必要**です。

本シリーズでは **sage-proteomics（MIT ライセンス、完全無料・商用可）** で代替します。

> **📝 INFO**
>
> **この記事で行う処理**
> DIA-MSの原理を理解し、sage-proteomicsのセットアップと設定を行います。DIA-MSデータをタンパク質定量マトリクスに変換するまでの全体像を学び、ヒトプロテオームのFASTAファイル準備からsage設定ファイル作成まで、実行準備を完了します。

In [ ]:
# 必要なライブラリをインポート
import os
import platform
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, HTML, Image
import json
import warnings
warnings.filterwarnings('ignore')

# プロジェクト設定
project_root = Path("/home/shizuku/labcode/article/Proteomics_drug_marker")
data_dir = project_root / "data" / "raw"
results_dir = project_root / "results" / "sage_output"

# ディレクトリ作成
results_dir.mkdir(parents=True, exist_ok=True)

print("🔮 sage-proteomics基礎学習開始")
print(f"📂 プロジェクトルート: {project_root}")
print(f"📁 データディレクトリ: {data_dir}")
print(f"📁 結果ディレクトリ: {results_dir}")
print(f"💻 OS: {platform.system()} {platform.release()}")

## 🌊 生データから定量マトリクスまで：処理の全体像

この章で行う処理を、「入力データ」「処理の中身」「出力データ」の3段階に分けて説明します。

In [ ]:
# データ処理の全体像を可視化
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# 1. 入力データ（mzMLファイル）
ax1.set_title('1. 入力：mzMLファイル（質量分析の生データ）', fontsize=12, fontweight='bold')
# スペクトラムのイメージ
mz_values = np.linspace(200, 1800, 100)
intensity1 = np.random.exponential(1, 100) * np.exp(-(mz_values-600)**2/50000)
intensity2 = np.random.exponential(1, 100) * np.exp(-(mz_values-900)**2/80000)
intensity3 = np.random.exponential(1, 100) * np.exp(-(mz_values-1200)**2/60000)

ax1.plot(mz_values, intensity1, alpha=0.7, label='スペクトル#1')
ax1.plot(mz_values, intensity2, alpha=0.7, label='スペクトル#2')
ax1.plot(mz_values, intensity3, alpha=0.7, label='スペクトル#3')
ax1.set_xlabel('m/z')
ax1.set_ylabel('Intensity')
ax1.legend()
ax1.text(200, max(intensity1)*0.8, '数千〜数万のスペクトラム\n各スペクトルに数百のピーク', 
         bbox=dict(boxstyle="round,pad=0.3", facecolor='lightblue', alpha=0.8))

# 2. FASTA入力（タンパク質データベース）
ax2.set_title('2. 補助入力：FASTAファイル（ヒト全タンパク質）', fontsize=12, fontweight='bold')
ax2.axis('off')
fasta_text = """>sp|P04637|P53_HUMAN
MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQ
AMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAP
>sp|P02768|ALBU_HUMAN  
MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKD
LGEEHFKGLVLIAFSQYLQQCPFDEHVKLVNELTEFW
...(約20,000タンパク質)"""
ax2.text(0.05, 0.7, fasta_text, fontsize=10, fontfamily='monospace',
         verticalalignment='top', transform=ax2.transAxes,
         bbox=dict(boxstyle="round,pad=0.3", facecolor='lightgreen', alpha=0.8))
ax2.text(0.05, 0.25, '各タンパク質のアミノ酸配列\n約20,000エントリ（ヒト）\nUniProt database', 
         transform=ax2.transAxes, fontsize=10,
         bbox=dict(boxstyle="round,pad=0.3", facecolor='lightgreen', alpha=0.5))

# 3. sage処理（同定・定量）
ax3.set_title('3. sage処理：同定と定量', fontsize=12, fontweight='bold')
processing_steps = [
    'FASTA → ペプチド断片生成\n(in silico消化)',
    'フラグメントイオン計算\n(理論スペクトル)',
    'スペクトル照合\n(実測vs理論)',
    'ペプチド同定\n(FDR<1%)',
    '定量計算\n(LFQ強度)'
]
y_positions = np.linspace(0.9, 0.1, len(processing_steps))
for i, (step, y) in enumerate(zip(processing_steps, y_positions)):
    ax3.text(0.1, y, f'{i+1}.', fontweight='bold', fontsize=12, 
             transform=ax3.transAxes)
    ax3.text(0.2, y, step, fontsize=10, transform=ax3.transAxes,
             bbox=dict(boxstyle="round,pad=0.2", facecolor='orange', alpha=0.3))
    if i < len(processing_steps) - 1:
        ax3.annotate('', xy=(0.15, y_positions[i+1] + 0.05), 
                    xytext=(0.15, y - 0.05),
                    arrowprops=dict(arrowstyle='->', lw=2, color='orange'),
                    transform=ax3.transAxes)
ax3.axis('off')

# 4. 出力（プロテインマトリクス）
ax4.set_title('4. 出力：タンパク質×サンプルの定量マトリクス', fontsize=12, fontweight='bold')
# サンプルデータマトリクスを表示
sample_data = np.random.lognormal(mean=15, sigma=1.5, size=(10, 8))
sample_data[np.random.random((10, 8)) < 0.15] = np.nan  # 欠損値を追加

im = ax4.imshow(sample_data, cmap='viridis', aspect='auto', interpolation='nearest')
ax4.set_xlabel('サンプル (CRC01-N, CRC01-T, ...)')
ax4.set_ylabel('タンパク質 (P53, ALBU, ...)')
ax4.set_xticks(range(8))
ax4.set_xticklabels(['CRC01-N', 'CRC01-T', 'CRC02-N', 'CRC02-T', 
                    'CRC03-N', 'CRC03-T', '...', 'CRC16-T'], rotation=45)
ax4.set_yticks(range(0, 10, 2))
ax4.set_yticklabels(['P53', 'ALBU', 'MYC', 'ACTB', '...'])

# カラーバーを追加
cbar = plt.colorbar(im, ax=ax4, shrink=0.8)
cbar.set_label('タンパク質強度 (log scale)')

ax4.text(3.5, -1.5, '2,110タンパク質 × 32サンプル\n統計解析・バイオマーカー探索に使用', 
         ha='center', fontsize=10,
         bbox=dict(boxstyle="round,pad=0.3", facecolor='lightcoral', alpha=0.8))

plt.tight_layout()
plt.show()

print("🔄 データ変換の流れ:")
print("1. mzML (数値データ) + FASTA (タンパク質配列)")
print("2. sage処理 (同定・定量)")
print("3. プロテインマトリクス (統計解析用)")
print("4. バイオマーカー発見 → 論文成果")

### 📊 入力：mzMLファイル（質量分析の生データ）とは

質量分析計に組織サンプルを入れると、サンプル中のペプチド（タンパク質の断片）が **質量/電荷比（m/z）** と **シグナル強度** のペアとして記録されます。これを「マススペクトル」と呼びます。

In [ ]:
# mzMLファイルの中身をシミュレート
def simulate_mzml_content():
    """mzMLファイルの構造と中身をシミュレート"""
    
    # 1つのスペクトルの例を生成
    spectrum_example = {
        "スペクトル番号": 1,
        "保持時間 (分)": 25.3,
        "MS level": "MS2",
        "precursor m/z": 524.78,
        "ピーク数": 247
    }
    
    # ピークデータの例（m/z, intensity）
    np.random.seed(42)
    peak_count = 10
    mz_values = np.sort(np.random.uniform(200, 1800, peak_count))
    intensities = np.random.exponential(5000, peak_count)
    
    peaks_df = pd.DataFrame({
        'm/z': mz_values,
        'intensity': intensities
    })
    
    print("📄 1つのmzMLファイルの中身（概念図）:")
    print("=" * 50)
    
    print(f"ファイル: CRC01-N.mzML (1.2GB)")
    print(f"総スペクトル数: 51,585個")
    print(f"測定時間: 0-120分")
    print(f"測定方式: DIA-MS (Data-Independent Acquisition)")
    print("")
    
    print("📊 スペクトル例:")
    for key, value in spectrum_example.items():
        print(f"  {key}: {value}")
    print("")
    
    print("📈 ピークデータ例（先頭10個）:")
    display(HTML(peaks_df.round(2).to_html(index=False)))
    
    print("\n💡 重要なポイント:")
    print("• この段階では「どのタンパク質がどれくらいあるか」は全くわからない")
    print("• m/zと強度の数値の羅列があるだけ")
    print("• DIA方式により、サンプル中のほぼすべてのペプチドを漏れなく記録")
    print("• 1ファイルで数万スペクトル、数百万ピークのデータ")
    
    return peaks_df

peaks_data = simulate_mzml_content()

### ⚙️ 処理：sageが行う「同定」と「定量」

sage は以下の手順で、m/zの数値データを「タンパク質名 × 強度」に変換します。

In [ ]:
# sageの処理ステップを詳細に説明
sage_processing_steps = {
    "ステップ": [
        "1. FASTA読み込み",
        "2. In silico消化",
        "3. フラグメント生成",
        "4. 理論スペクトル構築",
        "5. スペクトル照合",
        "6. ペプチド同定",
        "7. 定量計算 (LFQ)",
        "8. 結果出力"
    ],
    "入力": [
        "human_proteome.fasta",
        "約20,000タンパク質配列",
        "約300万ペプチド",
        "約6,300万フラグメント",
        "mzMLスペクトル",
        "照合済みスペクトル",
        "同定ペプチド",
        "LFQ強度"
    ],
    "処理内容": [
        "UniProt配列データの解析",
        "トリプシン酵素によるペプチド切断シミュレーション",
        "b/yイオンの理論m/z計算",
        "各ペプチドの予想スペクトルパターン作成",
        "実測スペクトルと理論スペクトルのマッチング",
        "統計的有意性検定(FDR<1%)",
        "ピーク面積統合とLabel-Free定量",
        "lfq.tsv, results.tsv等の生成"
    ],
    "出力": [
        "メモリ上のデータベース",
        "約300万ペプチド候補",
        "約6,300万理論フラグメント",
        "検索可能なスペクトルライブラリ",
        "PSM (Peptide-Spectrum Match)",
        "FDR<1%の高信頼ペプチド",
        "サンプル間定量値",
        "解析結果ファイル群"
    ]
}

steps_df = pd.DataFrame(sage_processing_steps)
display(HTML(steps_df.to_html(index=False, escape=False)))

print("\n🎯 重要なコンセプト:")
print("\n📚 In silico消化とは:")
print("• コンピュータ上でタンパク質をトリプシン酵素で切断")
print("• 実験で生じるペプチド断片を理論的に予測")
print("• 質量分析で検出されうるペプチドのリストを作成")

print("\n🔍 スペクトル照合とは:")
print("• 実測: mzMLファイル内のスペクトル")
print("• 理論: FASTAから計算したスペクトル")
print("• 照合: 十分な数のピークが一致すればペプチド同定")

print("\n📊 Label-Free Quantification (LFQ):")
print("• 同定されたペプチドのピーク面積を測定")
print("• サンプル間でタンパク質量を比較")
print("• 化学標識を使わない相対定量")

### 📈 出力：タンパク質×サンプルの定量マトリクス

sageの直接の出力はペプチドレベルの定量値（`lfq.tsv`）です。これを後段のスクリプトでタンパク質レベルに集約し、最終的に以下の形のCSVファイルが得られます。

In [ ]:
# プロテインマトリクスの構造を示すサンプルデータ
def create_sample_protein_matrix():
    """サンプルプロテインマトリクスを作成"""
    
    # サンプル名
    samples = []
    for i in range(1, 6):  # CRC01-05のみ表示
        samples.extend([f"CRC{i:02d}-N", f"CRC{i:02d}-T"])
    
    # タンパク質名（代表的ながん関連タンパク質）
    proteins = [
        "TP53_HUMAN (p53)",
        "KRAS_HUMAN (KRAS)", 
        "CTNNB1_HUMAN (β-catenin)",
        "PIK3CA_HUMAN (PI3K)",
        "ALBU_HUMAN (Albumin)",
        "ACTB_HUMAN (β-actin)",
        "MYC_HUMAN (c-Myc)",
        "EGFR_HUMAN (EGFR)"
    ]
    
    # データ生成（対数正規分布 + 欠損値）
    np.random.seed(42)
    n_proteins, n_samples = len(proteins), len(samples)
    
    # 基本的な強度値
    base_intensity = np.random.lognormal(mean=16, sigma=1.2, size=(n_proteins, n_samples))
    
    # がん関連タンパク質の腫瘍組織での上昇を模擬
    for i, protein in enumerate(proteins):
        if any(oncogene in protein for oncogene in ['KRAS', 'MYC', 'EGFR', 'PIK3CA']):
            # 腫瘍サンプル（奇数インデックス）で強度上昇
            base_intensity[i, 1::2] *= np.random.uniform(1.5, 3.0, len(samples)//2)
    
    # 欠損値の追加（15%程度）
    missing_mask = np.random.random((n_proteins, n_samples)) < 0.15
    matrix_data = base_intensity.copy()
    matrix_data[missing_mask] = np.nan
    
    # DataFrame作成
    protein_matrix = pd.DataFrame(matrix_data, index=proteins, columns=samples)
    
    return protein_matrix

# サンプルマトリクス作成と表示
sample_matrix = create_sample_protein_matrix()

print("📊 プロテインマトリクスの構造（抜粋）:")
print("=" * 60)
display(HTML(sample_matrix.round(2).to_html()))

print("\n📋 マトリクスの詳細:")
print(f"• 行数: {sample_matrix.shape[0]}タンパク質（実際は2,110個）")
print(f"• 列数: {sample_matrix.shape[1]}サンプル（実際は32サンプル）")
print(f"• 値: タンパク質の相対強度（log2スケール）")
print(f"• 欠損値: NaN（検出限界以下）")
print(f"• データ型: 数値行列（統計解析用）")

# 欠損値の統計
missing_percentage = (sample_matrix.isna().sum().sum() / sample_matrix.size) * 100
print(f"\n📉 欠損値統計: {missing_percentage:.1f}%")

print("\n🎯 このマトリクスの使用用途:")
print("✅ 前処理（Log2変換、欠損値補完）")
print("✅ 可視化（ヒートマップ、PCA、相関解析）")
print("✅ 統計検定（t検定、ANOVA、多重検定補正）")
print("✅ バイオマーカー探索（差分発現、パスウェイ解析）")
print("✅ COSMICデータベース照合（がん関連タンパク質同定）")

## 🛠️ ツール選択：なぜsage-proteomicsか

DIA-MSタンパク質同定ツールの比較と、商用利用の観点からsageを選択した理由を説明します。

In [ ]:
# DIA-MS解析ツールの比較
dia_tools_comparison = {
    "ツール名": [
        "DIA-NN",
        "FragPipe (MSFragger)",
        "Spectronaut",
        "sage-proteomics",
        "OpenMS + AlphaPeptDeep",
        "MaxQuant / MaxDIA"
    ],
    "ライセンス": [
        "学術無料 / 商用有料",
        "学術無料 / 商用有料",
        "商用有料",
        "MIT（完全無料）",
        "BSD + Apache 2.0",
        "非商用無料のみ"
    ],
    "商用利用": [
        "❌",
        "❌", 
        "⚠️ 有料",
        "✅ 完全無料",
        "✅ 完全無料",
        "⚠️ 制限あり"
    ],
    "本シリーズ採用": [
        "❌",
        "❌",
        "❌",
        "✅ メイン",
        "✅ 補助（深層学習）",
        "❌"
    ],
    "主な特徴": [
        "深層学習予測、論文標準",
        "統合プラットフォーム、多機能",
        "商用最高品質、企業標準",
        "軽量高速、Rust製、シンプル",
        "オープンソース、拡張性高",
        "MaxQuant統合、実績豊富"
    ],
    "検出性能目安": [
        "10,000+ (論文値)",
        "8,000-12,000",
        "10,000-15,000",
        "2,000-3,000 (本書)",
        "8,000-20,000 (期待値)",
        "3,000-8,000"
    ]
}

tools_df = pd.DataFrame(dia_tools_comparison)
display(HTML(tools_df.to_html(index=False, escape=False)))

print("\n🎯 sage-proteomicsを選択した理由:")
print("")
print("🏆 1. ライセンスの優位性")
print("  • MIT License → 商用利用完全OK")
print("  • 企業・出版物・商用製品での使用制限なし")
print("  • ライセンス料金・使用許諾申請が一切不要")
print("")
print("⚡ 2. 技術的優位性")
print("  • Rust製 → メモリ安全・高速実行")
print("  • 単一バイナリ → 依存関係なし、インストール簡単")
print("  • Library-free DIA → FASTAのみで完結")
print("  • 本データセット32ファイルを10.7分で解析")
print("")
print("🔬 3. 実用的優位性")
print("  • シンプル設定 → 学習コストが低い")
print("  • JSON設定ファイル → 可読性・再現性が高い")
print("  • アクティブ開発 → GitHub上で継続改善")
print("  • 論文実績 → J. Proteome Res. 2023 掲載")
print("")
print("⚠️ 4. 制限事項（正直な評価）")
print("  • 検出数: DIA-NNより少ない（2,110 vs 10,329）")
print("  • MBR未実装: Match Between Runs機能なし（将来実装予定）")
print("  • 深層学習未対応: スペクトル・保持時間予測なし")
print("")
print("💡 対策: OpenMS + AlphaPeptDeepとの併用で検出数向上")

## 📁 FASTAファイルの準備（ヒトプロテオーム配列）

sage はmzMLファイルだけでは動きません。「どのタンパク質が存在しうるか」の候補リストとして、**ヒト全タンパク質のアミノ酸配列を記録したFASTAファイル**が必要です。

In [ ]:
# FASTAファイルの構造と内容を説明
def explain_fasta_format():
    """FASTAファイルの構造を詳しく説明"""
    
    fasta_example = """>sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens GN=TP53 PE=1 SV=4
MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP
DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYPQGLNGTVNLPGRNSFEV
EDVVTLYHGKLLEAPQHKGGTTAYIAAPVTKRGGQKPPNKPKGRCGTTEPIIQLEKEEQR
KGPTIEQGRYGQPTVQTIIWKKDDPPYKPYHFWTCAIFSKVLEDEKYACRLPNTKPGTG
>sp|P02768|ALBU_HUMAN Serum albumin OS=Homo sapiens GN=ALB PE=1 SV=2
MKWVTFISLLLLFSSAYSRGVFRRDTHKSEIAHRFKDLGEEHFKGLVLIAFSQYLQQCPF
DEHVKLVNELTEFAKTCVADESHAGCEKSLHTLFGDELCKVASLRETYGDMADCCEKQEP
ERNECFLSHKDDSPDLPKLKPDPNTLCDEFKADEKKFWGKYLYEIARRHPYFYAPELLYY
..."""
    
    print("📄 FASTAファイルの構造:")
    print("=" * 50)
    print(fasta_example)
    print("")
    
    print("🔍 ヘッダ行の解読:")
    print(">sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens GN=TP53 PE=1 SV=4")
    print("")
    
    header_components = {
        "要素": [
            "sp|",
            "P04637",
            "P53_HUMAN",
            "Cellular tumor antigen p53",
            "OS=Homo sapiens",
            "GN=TP53",
            "PE=1",
            "SV=4"
        ],
        "意味": [
            "SwissProt database",
            "UniProt Accession ID（一意識別子）",
            "Entry name（データベース内名称）",
            "Protein name（タンパク質名）",
            "Organism species（生物種）",
            "Gene name（遺伝子シンボル）★重要",
            "Protein existence（証拠レベル1-5）",
            "Sequence version（配列バージョン）"
        ],
        "sageでの使用": [
            "データベース種別識別",
            "一意識別（重複除去）",
            "表示用ラベル",
            "アノテーション",
            "種特異性確認",
            "遺伝子名変換（重要）",
            "品質フィルタ",
            "バージョン管理"
        ]
    }
    
    header_df = pd.DataFrame(header_components)
    display(HTML(header_df.to_html(index=False, escape=False)))
    
    print("\n🧬 配列部分の特徴:")
    print("• 1文字アミノ酸コード（A, R, N, D, C, ...）")
    print("• 60文字で改行（可読性のため）")
    print("• 配列長: 数十〜数千アミノ酸")
    print("• 修飾残基や異常アミノ酸も含む")
    
    print("\n📊 ヒトプロテオームの統計:")
    human_proteome_stats = {
        "項目": ["総エントリ数", "reviewed (Swiss-Prot)", "unreviewed (TrEMBL)", "平均配列長", "最短配列", "最長配列"],
        "値": ["約200,000", "約20,000 (高品質)", "約180,000", "約500 aa", "約10 aa", "約35,000 aa"],
        "sage使用": ["全体", "推奨（本書で使用）", "補助", "-", "-", "-"]
    }
    stats_df = pd.DataFrame(human_proteome_stats)
    display(HTML(stats_df.to_html(index=False, escape=False)))
    
    print("\n⭐ 重要: GN=（遺伝子名）の役割")
    print("• sage出力はUniProt IDで記録")
    print("• 後段でGN=の値を使って遺伝子シンボルに変換")
    print("• TP53, KRAS, CTNNB1等の馴染みある名前で解析結果表示")
    print("• COSMICデータベースとの照合にも遺伝子シンボルを使用")

explain_fasta_format()

In [ ]:
# FASTAファイルのダウンロード方法
def show_fasta_download_methods():
    """FASTAファイルのダウンロード方法を表示"""
    
    download_methods = {
        "方法": [
            "ブラウザ（推奨）",
            "curl/wget", 
            "Python requests",
            "BioPython"
        ],
        "難易度": [
            "易",
            "中",
            "中",
            "上"
        ],
        "信頼性": [
            "高",
            "高",
            "中",
            "高"
        ],
        "自動化": [
            "手動",
            "可能",
            "可能",
            "可能"
        ]
    }
    
    methods_df = pd.DataFrame(download_methods)
    display(HTML(methods_df.to_html(index=False, escape=False)))
    
    print("\n🌐 方法1: ブラウザ（推奨・確実）")
    print("1. https://www.uniprot.org/uniprotkb?query=reviewed:true+AND+organism_id:9606 にアクセス")
    print("2. ページ上部の 'Download' ボタンをクリック")
    print("3. Format で 'FASTA (canonical)' を選択")
    print("4. 'Download' でファイル保存")
    
    print("\n💻 方法2: コマンドライン（自動化）")
    print("```bash")
    print("# UniProtからヒト reviewed プロテオームをダウンロード")
    print("curl -o data/raw/human_proteome.fasta \\")
    print('  "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=reviewed:true+AND+organism_id:9606"')
    print("")
    print("# 確認: エントリ数を表示")
    print("grep -c '^>' data/raw/human_proteome.fasta")
    print("# → 約 20,000 エントリ")
    print("```")
    
    print("\n🐍 方法3: Python（Notebook内で実行可能）")
    print("```python")
    print("import requests")
    print("from pathlib import Path")
    print("")
    print("url = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=reviewed:true+AND+organism_id:9606'")
    print("fasta_path = Path('data/raw/human_proteome.fasta')")
    print("")
    print("response = requests.get(url)")
    print("if response.status_code == 200:")
    print("    fasta_path.parent.mkdir(parents=True, exist_ok=True)")
    print("    fasta_path.write_text(response.text)")
    print("    print(f'Downloaded: {fasta_path}')")
    print("```")
    
    print("\n⚠️ 重要な注意点:")
    print("• ファイルサイズ: 約10-15MB")
    print("• 更新頻度: 月1回程度（エントリ数が少し変動）")
    print("• ネットワーク: 安定した接続が必要")
    print("• バージョン: 論文再現には同一バージョン推奨")

show_fasta_download_methods()

## ⚙️ sage-proteomicsのセットアップ

sage-proteomicsのインストールと基本動作確認を行います。

In [ ]:
# sage-proteomicsの基本情報
sage_info = {
    "項目": [
        "開発者",
        "初版",
        "プログラミング言語",
        "ライセンス",
        "DIA サポート",
        "論文",
        "GitHub",
        "最新バージョン",
        "インストール方法"
    ],
    "内容": [
        "Michael Lazear",
        "2022年",
        "Rust（単一バイナリ、依存なし）",
        "MIT（商用完全OK）",
        "✅ library-free DIA ネイティブ対応",
        "Lazear MR. J. Proteome Res. 2023. DOI: 10.1021/acs.jproteome.3c00486",
        "https://github.com/lazear/sage",
        "0.14.6+ (2024年)",
        "conda, bioconda, バイナリ直接ダウンロード"
    ]
}

sage_df = pd.DataFrame(sage_info)
display(HTML(sage_df.to_html(index=False, escape=False)))

print("\n🚀 sageの強み:")
print("")
print("⚡ 1. 高速性能")
print("  • Rust製 → ネイティブ最適化、ゼロコスト抽象化")
print("  • 並列処理 → マルチコアCPUを効率活用")
print("  • 本データセット32ファイルを10.7分で解析")
print("")
print("🔧 2. 導入簡単")
print("  • 単一バイナリ → 依存関係なし")
print("  • クロスプラットフォーム → Windows/Linux/macOS対応")
print("  • conda-forgeから簡単インストール")
print("")
print("🎯 3. Library-free DIA")
print("  • FASTAファイルのみで完結")
print("  • スペクトラムライブラリ構築不要")
print("  • In silico消化から定量まで一貫処理")
print("")
print("⚖️ 4. ライセンス")
print("  • MIT License → 商用利用制限なし")
print("  • ソースコード公開 → 透明性・再現性")
print("  • アクティブ開発 → 継続的改善")

In [ ]:
# sage-proteomicsのインストール確認
def check_sage_installation():
    """sageのインストール状況を確認"""
    
    try:
        # sageコマンドの存在確認
        result = subprocess.run(['sage', '--version'], 
                              capture_output=True, text=True, timeout=10)
        
        if result.returncode == 0:
            version = result.stdout.strip()
            print(f"✅ sage-proteomics がインストール済み: {version}")
            
            # ヘルプの一部を表示
            help_result = subprocess.run(['sage', '--help'], 
                                       capture_output=True, text=True, timeout=10)
            if help_result.returncode == 0:
                help_lines = help_result.stdout.split('\n')[:15]
                print("\n📖 sage ヘルプ（抜粋）:")
                for line in help_lines:
                    if line.strip():
                        print(f"  {line}")
            
            return True
            
        else:
            print("❌ sage コマンドでエラー")
            print(result.stderr)
            return False
            
    except FileNotFoundError:
        print("❌ sage-proteomics がインストールされていません")
        return False
    except subprocess.TimeoutExpired:
        print("❌ sage コマンドがタイムアウト")
        return False
    except Exception as e:
        print(f"❌ sage 確認エラー: {e}")
        return False

sage_installed = check_sage_installation()

if not sage_installed:
    print("\n🔧 sage-proteomicsのインストール方法:")
    print("")
    print("方法1: conda/micromamba（推奨）")
    print("```bash")
    print("micromamba activate crc-proteomics")
    print("micromamba install -c bioconda -c conda-forge sage-proteomics -y")
    print("sage --version")
    print("```")
    print("")
    print("方法2: バイナリ直接ダウンロード")
    print("1. https://github.com/lazear/sage/releases にアクセス")
    print("2. 最新版のバイナリをダウンロード（OS別）")
    print("3. PATHの通ったディレクトリに配置")
    print("")
    print("方法3: Cargo（Rust開発環境）")
    print("```bash")
    print("cargo install sage-proteomics")
    print("```")
else:
    print("\n🎉 sage-proteomicsの準備完了！")

## ⚙️ sage設定ファイル (sage_config.json)

論文の DIA-NN パラメータに合わせた設定ファイルを作成します。

In [ ]:
# sage設定ファイルの作成
def create_sage_config():
    """論文に合わせたsage設定ファイルを作成"""
    
    sage_config = {
        "database": {
            "bucket_size": 32768,
            "fragment_min_mz": 200.0,
            "fragment_max_mz": 1800.0,
            "peptide_min_mass": 600.0,
            "peptide_max_mass": 4000.0,
            "enzyme": {
                "missed_cleavages": 1,
                "min_len": 7,
                "max_len": 45,
                "cleave_at": "KR",
                "restrict": "P",
                "c_terminal": True
            },
            "static_mods": {"C": 57.02146},
            "variable_mods": {},
            "max_variable_mods": 1,
            "ion_kinds": ["b", "y"],
            "decoy_tag": "rev_",
            "generate_decoys": True,
            "fasta": "data/raw/human_proteome.fasta"
        },
        "quant": {
            "lfq": True,
            "lfq_settings": {
                "peak_scoring": "Hybrid",
                "integration": "Sum",
                "spectral_angle": 0.7,
                "ppm_tolerance": 10.0,
                "combine_charge_states": True
            }
        },
        "precursor_tol": {"ppm": [-10, 10]},
        "fragment_tol": {"ppm": [-10, 10]},
        "precursor_charge": [2, 4],
        "isotope_errors": [0, 0],
        "deisotope": False,
        "chimera": True,
        "wide_window": True,
        "predict_rt": False,
        "min_peaks": 15,
        "max_peaks": 150,
        "min_matched_peaks": 4,
        "max_fragment_charge": 2,
        "report_psms": 1,
        "output_directory": "results/sage_output"
    }
    
    return sage_config

# 設定ファイルを作成
config = create_sage_config()
config_path = project_root / "sage_config.json"

# JSON形式で保存
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("📝 sage設定ファイルを作成しました:")
print(f"保存先: {config_path}")
print("")
print("⚙️ 設定内容の表示:")
print(json.dumps(config, indent=2, ensure_ascii=False)[:1000] + "...")

print(f"\n✅ 完全な設定ファイルが {config_path} に保存されました")

In [ ]:
# 論文のDIA-NNパラメータとsageパラメータの対応を詳細説明
parameter_mapping = {
    "論文（DIA-NN）パラメータ": [
        "--fasta-search（library-free）",
        "--cut K*,R*（トリプシン消化）",
        "--missed-cleavages 1",
        "--min-pep-len 7",
        "--max-pep-len 45",
        "--pr-charges 2-4",
        "--min-fr-mz 200 / --max-fr-mz 1800",
        "MS1/MS2 accuracy 10 ppm",
        "Cys carbamidomethylation",
        "FDR <1%",
        "MBR (Match Between Runs)"
    ],
    "sage対応パラメータ": [
        'wide_window: true + FASTA指定',
        'enzyme.cleave_at: "KR"',
        'enzyme.missed_cleavages: 1',
        'enzyme.min_len: 7',
        'enzyme.max_len: 45',
        'precursor_charge: [2, 4]',
        'fragment_min_mz: 200 / fragment_max_mz: 1800',
        'precursor_tol.ppm: [-10,10] / fragment_tol.ppm',
        'static_mods: { "C": 57.02146 }',
        '出力の peptide_q < 0.01 で後段フィルタ',
        'sage 0.14では未実装（将来実装予定）'
    ],
    "設定値の意味": [
        "FASTAから理論スペクトル生成、ライブラリ不要",
        "リジン(K)・アルギニン(R)でペプチド切断",
        "切断ミス1回まで許容",
        "最短ペプチド長7アミノ酸",
        "最長ペプチド長45アミノ酸",
        "プリカーサーイオン電荷+2〜+4",
        "フラグメントイオンm/z範囲200-1800",
        "質量精度±10ppm以内",
        "システインのカルバミドメチル化（+57Da）",
        "偽陽性発見率1%未満",
        "サンプル間ピーク照合（検出数向上）"
    ],
    "重要度": [
        "必須（DIA解析の基本）",
        "必須（酵素特異性）",
        "重要（現実的消化）",
        "重要（検出感度）",
        "重要（計算効率）",
        "必須（イオン化特性）",
        "重要（装置特性）",
        "必須（同定精度）",
        "必須（前処理反映）",
        "必須（信頼性保証）",
        "オプション（性能向上）"
    ]
}

mapping_df = pd.DataFrame(parameter_mapping)
display(HTML(mapping_df.to_html(index=False, escape=False)))

print("\n🎯 重要パラメータの詳細解説:")
print("")
print("🔬 wide_window: true")
print("  • DIA-MSの本質的設定")
print("  • 広いm/z窓での検索を有効化")
print("  • 複数ペプチドの混合スペクトルに対応")
print("")
print("🧬 enzyme設定（トリプシン特異性）")
print("  • cleave_at: 'KR' → リジン・アルギニンで切断")
print("  • restrict: 'P' → プロリンの次は切断しない")
print("  • c_terminal: true → C末端での切断")
print("")
print("⚗️ static_mods（固定修飾）")
print("  • C: 57.02146 → カルバミドメチル化")
print("  • サンプル調製時の化学修飾を反映")
print("  • すべてのシステインに適用")
print("")
print("📊 LFQ設定（Label-Free定量）")
print("  • peak_scoring: 'Hybrid' → 複合スコアリング")
print("  • integration: 'Sum' → ピーク面積積分")
print("  • spectral_angle: 0.7 → スペクトル類似度閾値")

## 🎯 まとめと次のステップ

DIA-MSプロテオミクス解析の原理を理解し、sage-proteomicsのセットアップが完了しました。

In [ ]:
# 完了チェックリストと次のステップ
completion_checklist = {
    "完了項目": [
        "DIA-MS理論理解",
        "データ処理の全体像把握", 
        "sage-proteomics概要",
        "FASTAファイル準備方法",
        "sage設定ファイル作成",
        "パラメータ対応確認",
        "次章への準備"
    ],
    "習得内容": [
        "mzML→プロテインマトリクス変換プロセス",
        "In silico消化・スペクトル照合・LFQ定量",
        "MIT License・高速・Library-free特徴",
        "UniProt・FASTA形式・ヒトプロテオーム",
        "DIA-NN対応・JSON設定・最適化",
        "論文再現のためのパラメータ調整",
        "sage実行環境の整備完了"
    ],
    "期待成果": [
        "解析原理の明確な理解",
        "データ流れの可視化",
        "ツール選択の合理性",
        "データベース準備完了",
        "実行可能な設定",
        "論文との整合性",
        "実解析への準備完了"
    ]
}

checklist_df = pd.DataFrame(completion_checklist)
display(HTML(checklist_df.to_html(index=False, escape=False)))

print("\n🚀 次章での実行内容:")
print("")
print("📂 [#4b sage実行と結果解析](notebook_04b_sage_execution.ipynb):")
print("  1. 32個のmzMLファイルをsageで一括解析")
print("  2. ペプチドレベル結果（lfq.tsv）の取得")
print("  3. タンパク質レベルの定量マトリクス生成")
print("  4. 2,110タンパク質 × 32サンプルの解析結果")
print("  5. 遺伝子シンボル変換とアノテーション")
print("")
print("📊 期待される成果:")
print(f"  • プロテインマトリクス: results/protein_matrix_from_sage.csv")
print(f"  • 検出タンパク質数: 約2,110個")
print(f"  • 処理時間: 約10-15分（32ファイル）")
print(f"  • 次章以降の統計解析・可視化の入力データ完成")

print("\n✅ 準備完了確認:")
required_items = [
    ("mzMLファイル", "32個のDIA-MSデータ"),
    ("FASTAファイル", "ヒト全タンパク質配列"),
    ("sage設定ファイル", "sage_config.json"),
    ("sage-proteomics", "実行環境"),
    ("出力ディレクトリ", "results/sage_output")
]

for item, description in required_items:
    print(f"  📋 {item}: {description}")

print("\n🎉 すべて準備完了！sage実行開始の準備が整いました")

print("\n#バイオインフォマティクス #プロテオミクス #sage #DIA-MS #labcode")